In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/areejfatimaa223/bseline-and-afa/afa_latest.pt


In [2]:
from torchvision import models
import torch.nn as nn
import torch

afa_model = models.resnet18(weights=None)
afa_model.fc = nn.Linear(afa_model.fc.in_features, 10)
afa_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
afa_model.maxpool = nn.Identity()
afa_model = convert_to_dualbn(afa_model)  # your DualBN2d + convert_to_dualbn function from earlier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
afa_model = afa_model.to(device)
print("Using device:", device)

NameError: name 'convert_to_dualbn' is not defined

In [4]:
import torch.nn as nn

class DualBN2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn_main = nn.BatchNorm2d(num_features)
        self.bn_aux = nn.BatchNorm2d(num_features)
        self.branch = 'main'

    def forward(self, x):
        if self.branch == 'main':
            return self.bn_main(x)
        else:
            return self.bn_aux(x)

def convert_to_dualbn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, DualBN2d(child.num_features))
        else:
            convert_to_dualbn(child)
    return module

def set_branch(model, branch):
    for m in model.modules():
        if isinstance(m, DualBN2d):
            m.branch = branch

In [5]:
from torchvision import models
import torch.nn as nn
import torch

afa_model = models.resnet18(weights=None)
afa_model.fc = nn.Linear(afa_model.fc.in_features, 10)
afa_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
afa_model.maxpool = nn.Identity()
afa_model = convert_to_dualbn(afa_model)  # your DualBN2d + convert_to_dualbn function from earlier

device = 'cuda' if torch.cuda.is_available() else 'cpu'
afa_model = afa_model.to(device)
print("Using device:", device)

Using device: cuda


In [6]:
checkpoint_path = "/kaggle/input/datasets/areejfatimaa223/bseline-and-afa/afa_latest.pt"
afa_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("Checkpoint loaded successfully")

RuntimeError: Error(s) in loading state_dict for ResNet:
	size mismatch for conv1.weight: copying a param with shape torch.Size([64, 3, 7, 7]) from checkpoint, the shape in current model is torch.Size([64, 3, 3, 3]).

In [7]:
afa_model = models.resnet18(weights=None)
afa_model.fc = nn.Linear(afa_model.fc.in_features, 10)

# CIFAR fix — confirm this is actually here this time
afa_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
afa_model.maxpool = nn.Identity()

afa_model = convert_to_dualbn(afa_model)
afa_model = afa_model.to(device)

# verify before training
print(afa_model.conv1)   # should print kernel_size=(3, 3)
print(afa_model.maxpool) # should print Identity()

Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Identity()


In [8]:
afa_optimizer = torch.optim.SGD(afa_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
afa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(afa_optimizer, T_max=NUM_EPOCHS)

NameError: name 'NUM_EPOCHS' is not defined

In [9]:
NUM_EPOCHS = 60

afa_optimizer = torch.optim.SGD(afa_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
afa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(afa_optimizer, T_max=NUM_EPOCHS)

In [10]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
test_transform = transforms.Compose([transforms.ToTensor()])

DATA_DIR = "/kaggle/working/data"
train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [16:59<00:00, 167kB/s] 


In [11]:
import os
import json
import time
import math
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cuda


In [12]:
class DualBN2d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.bn_main = nn.BatchNorm2d(num_features)
        self.bn_aux = nn.BatchNorm2d(num_features)
        self.branch = 'main'
 
    def forward(self, x):
        if self.branch == 'main':
            return self.bn_main(x)
        else:
            return self.bn_aux(x)
 
 
def convert_to_dualbn(module):
    """Recursively replace every BatchNorm2d in the model with DualBN2d."""
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, DualBN2d(child.num_features))
        else:
            convert_to_dualbn(child)
    return module
 
 
def set_branch(model, branch):
    """Set the active branch ('main' or 'aux') on every DualBN2d layer."""
    assert branch in ('main', 'aux')
    for m in model.modules():
        if isinstance(m, DualBN2d):
            m.branch = branch
 

In [13]:
def fourier_basis(H, W, f, omega, device):
    u = torch.arange(H, device=device).view(H, 1).float()
    v = torch.arange(W, device=device).view(1, W).float()
    wave = torch.sin(2 * math.pi * f * (u * math.cos(omega) + v * math.sin(omega)) - math.pi / 4)
    wave = wave / wave.norm(p=2)  # unit L2 norm, the paper's "R"
    return wave
 
 
def afa_augment(images, inv_lambda=8.0):
    """
    images: (B, C, H, W) tensor in [0, 1]
    inv_lambda: mean of the exponential distribution the per-channel
                strength is sampled from (paper's 1/lambda).
    """
    B, C, H, W = images.shape
    dev = images.device
    M = max(H, W)
    out = images.clone()
    exp_dist = torch.distributions.Exponential(1.0 / inv_lambda)
 
    for b in range(B):
        for c in range(C):
            f = torch.empty(1).uniform_(1, M).item()
            omega = torch.empty(1).uniform_(0, math.pi).item()
            sigma = exp_dist.sample().item()
            wave = fourier_basis(H, W, f, omega, dev)
            out[b, c] = images[b, c] + sigma * wave
 
    return torch.clamp(out, 0.0, 1.0)


In [14]:
NUM_EPOCHS = 60  # adjust to your time budget; note any change from the paper's 200 in your report
 
afa_model = models.resnet18(weights=None)
afa_model.fc = nn.Linear(afa_model.fc.in_features, 10)
 
# CIFAR fix: 3x3 stride-1 stem, no maxpool — do NOT skip this
afa_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
afa_model.maxpool = nn.Identity()
 
afa_model = convert_to_dualbn(afa_model)
afa_model = afa_model.to(device)
 
# verify the fix actually landed before training
print(afa_model.conv1)    # expect kernel_size=(3, 3), stride=(1, 1)
print(afa_model.maxpool)  # expect Identity()
 
n_dual = sum(1 for m in afa_model.modules() if isinstance(m, DualBN2d))
print(f"Converted {n_dual} BatchNorm layers to DualBN2d")
 
 
# %% CELL 5 — optimizer, scheduler, loss
afa_optimizer = torch.optim.SGD(afa_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
afa_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(afa_optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
 

Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Identity()
Converted 20 BatchNorm layers to DualBN2d


In [15]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
test_transform = transforms.Compose([transforms.ToTensor()])
 
DATA_DIR = "/kaggle/working/data"
 
train_dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform)
test_dataset = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)
 
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
 
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([64, 3, 32, 32])
Label batch shape: torch.Size([64])


In [16]:
def train_step(images, labels):
    images, labels = images.to(device), labels.to(device)
    afa_images = afa_augment(images, inv_lambda=8.0)
 
    afa_optimizer.zero_grad()
 
    set_branch(afa_model, 'main')
    logits_main = afa_model(images)
    loss_main = criterion(logits_main, labels)
 
    set_branch(afa_model, 'aux')
    logits_aux = afa_model(afa_images)
    loss_aux = criterion(logits_aux, labels)
 
    ace_loss = 0.5 * (loss_main + loss_aux)
    ace_loss.backward()
    afa_optimizer.step()
 
    return ace_loss.item(), loss_main.item(), loss_aux.item()
 
 
def evaluate(model, loader, branch=None):
    model.eval()
    if branch is not None:
        set_branch(model, branch)
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    model.train()
    return 100 * correct / total
 
 
# quick sanity check on one batch before committing to the full run
images, labels = next(iter(train_loader))
ace, m, a = train_step(images, labels)
print(f"Sanity check — ACE: {ace:.4f}, main: {m:.4f}, aux: {a:.4f}")
 

Sanity check — ACE: 2.5841, main: 2.5898, aux: 2.5784


In [17]:
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RESULTS_DIR = "/kaggle/working/results"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
 
afa_log = []
 
for epoch in range(NUM_EPOCHS):
    start = time.time()
    running_ace, running_main, running_aux = 0.0, 0.0, 0.0
 
    for images, labels in train_loader:
        ace, m, a = train_step(images, labels)
        running_ace += ace
        running_main += m
        running_aux += a
 
    afa_scheduler.step()
    n = len(train_loader)
    avg_ace, avg_main, avg_aux = running_ace / n, running_main / n, running_aux / n
    elapsed = time.time() - start
 
    log_entry = {
        "epoch": epoch + 1,
        "ace_loss": avg_ace,
        "main_loss": avg_main,
        "aux_loss": avg_aux,
        "time_sec": elapsed,
    }
 
    if (epoch + 1) % 10 == 0 or epoch == NUM_EPOCHS - 1:
        acc = evaluate(afa_model, test_loader, branch='main')  # paper evaluates main branch only at test time
        log_entry["test_acc"] = acc
        print(f"[AFA] Epoch {epoch+1}/{NUM_EPOCHS} | ACE {avg_ace:.4f} | test acc {acc:.2f}% | {elapsed:.1f}s")
    else:
        print(f"[AFA] Epoch {epoch+1}/{NUM_EPOCHS} | ACE {avg_ace:.4f} | {elapsed:.1f}s")
 
    afa_log.append(log_entry)
 
    torch.save(afa_model.state_dict(), f"{CHECKPOINT_DIR}/afa_latest.pt")
    with open(f"{RESULTS_DIR}/afa_log.json", "w") as f:
        json.dump(afa_log, f, indent=2)
 
print("AFA training complete.")

[AFA] Epoch 1/60 | ACE 2.0405 | 134.7s
[AFA] Epoch 2/60 | ACE 1.6309 | 134.1s
[AFA] Epoch 3/60 | ACE 1.3835 | 134.3s
[AFA] Epoch 4/60 | ACE 1.1411 | 134.1s
[AFA] Epoch 5/60 | ACE 0.9755 | 133.9s
[AFA] Epoch 6/60 | ACE 0.9016 | 133.7s
[AFA] Epoch 7/60 | ACE 0.8484 | 133.8s
[AFA] Epoch 8/60 | ACE 0.8170 | 133.9s
[AFA] Epoch 9/60 | ACE 0.7815 | 134.2s
[AFA] Epoch 10/60 | ACE 0.7578 | test acc 80.35% | 134.1s
[AFA] Epoch 11/60 | ACE 0.7410 | 133.8s
[AFA] Epoch 12/60 | ACE 0.7195 | 133.8s
[AFA] Epoch 13/60 | ACE 0.7104 | 133.8s
[AFA] Epoch 14/60 | ACE 0.6917 | 133.6s
[AFA] Epoch 15/60 | ACE 0.6804 | 133.4s
[AFA] Epoch 16/60 | ACE 0.6676 | 133.8s
[AFA] Epoch 17/60 | ACE 0.6552 | 133.3s
[AFA] Epoch 18/60 | ACE 0.6441 | 133.6s
[AFA] Epoch 19/60 | ACE 0.6300 | 133.7s
[AFA] Epoch 20/60 | ACE 0.6255 | test acc 81.05% | 133.4s
[AFA] Epoch 21/60 | ACE 0.6125 | 133.3s
[AFA] Epoch 22/60 | ACE 0.6002 | 133.9s
[AFA] Epoch 23/60 | ACE 0.5907 | 133.8s
[AFA] Epoch 24/60 | ACE 0.5724 | 134.0s
[AFA] Epoch 2

KeyboardInterrupt: 

In [18]:
import os
print(os.path.exists("/kaggle/working/checkpoints/afa_latest.pt"))
print(os.path.exists("/kaggle/working/results/afa_log.json"))

True
True


In [19]:
import os
print(os.path.exists("/kaggle/working/checkpoints"))
print(os.listdir("/kaggle/working"))
if os.path.exists("/kaggle/working/checkpoints"):
    print(os.listdir("/kaggle/working/checkpoints"))

True
['results', 'checkpoints', '.ipynb_checkpoints', 'data', '.virtual_documents']
['afa_latest.pt']


In [20]:
from IPython.display import FileLink
FileLink("/kaggle/working/checkpoints/afa_latest.pt")

/kaggle/working/checkpoints/afa_latest.pt

In [21]:
import shutil
shutil.make_archive('/kaggle/working/afa_checkpoint', 'zip', '/kaggle/working/checkpoints')
print("Zipped successfully")

Zipped successfully


In [22]:
shutil.make_archive('/kaggle/working/afa_results', 'zip', '/kaggle/working/results')
print("Results zipped")

Results zipped


In [24]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if 'cifar' in root.lower() and f.endswith('.npy'):
            print(os.path.join(root, f))

/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/motion_blur.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/speckle_noise.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/jpeg_compression.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/impulse_noise.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/defocus_blur.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/brightness.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/elastic_transform.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/snow.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/labels.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/gaussian_blur.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/frost.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/zoom_blur.npy
/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C/spatter.npy
/kaggle/input/datasets/harshadak

In [25]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.pt'):
            print(os.path.join(root, f))

/kaggle/input/datasets/areejfatimaa223/baseline/baseline_latest.pt
/kaggle/input/datasets/areejfatimaa223/bseline-and-afa/afa_latest.pt


In [26]:
baseline_model = models.resnet18(weights=None)
baseline_model.fc = nn.Linear(baseline_model.fc.in_features, 10)
baseline_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
baseline_model.maxpool = nn.Identity()
baseline_model = baseline_model.to(device)

baseline_checkpoint_path = "<paste the baseline .pt path found above>"
baseline_model.load_state_dict(torch.load(baseline_checkpoint_path, map_location=device))
baseline_model.eval()
print("Baseline checkpoint loaded")

FileNotFoundError: [Errno 2] No such file or directory: '<paste the baseline .pt path found above>'

In [27]:
baseline_model = models.resnet18(weights=None)
baseline_model.fc = nn.Linear(baseline_model.fc.in_features, 10)
baseline_model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
baseline_model.maxpool = nn.Identity()
baseline_model = baseline_model.to(device)

baseline_checkpoint_path = "/kaggle/input/datasets/areejfatimaa223/baseline/baseline_latest.pt"
baseline_model.load_state_dict(torch.load(baseline_checkpoint_path, map_location=device))
baseline_model.eval()
print("Baseline checkpoint loaded")

Baseline checkpoint loaded


In [28]:
import numpy as np

CIFAR10C_DIR = "/kaggle/input/datasets/harshadakhatu/cifar-10-c/CIFAR-10-C"

corruptions = [
    "gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur",
    "glass_blur", "motion_blur", "zoom_blur", "snow", "frost", "fog",
    "brightness", "contrast", "elastic_transform", "pixelate",
    "jpeg_compression", "speckle_noise", "gaussian_blur", "spatter", "saturate"
]

labels_c = np.load(os.path.join(CIFAR10C_DIR, "labels.npy"))

def evaluate_on_corruption(model, corruption_name, batch_size=256, branch=None):
    images_c = np.load(os.path.join(CIFAR10C_DIR, f"{corruption_name}.npy"))
    model.eval()
    if branch is not None:
        set_branch(model, branch)

    correct, total = 0, 0
    with torch.no_grad():
        for i in range(0, len(images_c), batch_size):
            batch_imgs = images_c[i:i+batch_size]
            batch_labels = labels_c[i:i+batch_size]

            imgs = torch.from_numpy(batch_imgs).float().permute(0, 3, 1, 2) / 255.0
            imgs = imgs.to(device)
            lbls = torch.from_numpy(batch_labels).long().to(device)

            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)

    return 100 * correct / total

In [29]:
results = {"baseline": {}, "afa": {}}

for c in corruptions:
    acc_baseline = evaluate_on_corruption(baseline_model, c)
    acc_afa = evaluate_on_corruption(afa_model, c, branch='main')
    results["baseline"][c] = acc_baseline
    results["afa"][c] = acc_afa
    print(f"{c:20s} | baseline: {acc_baseline:5.2f}% | AFA: {acc_afa:5.2f}%")

ra_baseline = sum(results["baseline"].values()) / len(corruptions)
ra_afa = sum(results["afa"].values()) / len(corruptions)
print(f"\nOverall Robustness Accuracy (RA):")
print(f"Baseline: {ra_baseline:.2f}%")
print(f"AFA:      {ra_afa:.2f}%")

import json
with open("/kaggle/working/results/cifar10c_results.json", "w") as f:
    json.dump(results, f, indent=2)

gaussian_noise       | baseline: 48.34% | AFA: 61.47%
shot_noise           | baseline: 60.12% | AFA: 67.07%
impulse_noise        | baseline: 53.62% | AFA: 59.90%
defocus_blur         | baseline: 81.30% | AFA: 63.94%
glass_blur           | baseline: 54.54% | AFA: 51.60%
motion_blur          | baseline: 78.61% | AFA: 56.59%
zoom_blur            | baseline: 77.32% | AFA: 54.77%
snow                 | baseline: 82.36% | AFA: 69.22%
frost                | baseline: 76.75% | AFA: 65.66%
fog                  | baseline: 88.15% | AFA: 67.49%
brightness           | baseline: 93.23% | AFA: 78.96%
contrast             | baseline: 77.87% | AFA: 54.31%
elastic_transform    | baseline: 83.73% | AFA: 67.64%
pixelate             | baseline: 74.78% | AFA: 62.03%
jpeg_compression     | baseline: 79.05% | AFA: 70.04%
speckle_noise        | baseline: 63.16% | AFA: 68.26%
gaussian_blur        | baseline: 70.32% | AFA: 54.78%
spatter              | baseline: 83.78% | AFA: 71.91%
saturate             | basel

In [30]:
!git clone https://github.com/SoyatoyaCodes/afa-augment_reproduction.git
%cd afa-augment_reproduction

Cloning into 'afa-augment_reproduction'...
remote: Enumerating objects: 316, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 316 (delta 18), reused 1 (delta 1), pack-reused 262 (from 1)
Receiving objects: 100% (316/316), 2.04 MiB | 13.49 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/kaggle/working/afa-augment_reproduction


In [31]:
!git config --global user.name "syedaareejfatima223-stack"
!git config --global user.email "syedaareejfatima223@gmail.com"

In [32]:
!git checkout -b areej-afa-work

Switched to a new branch 'areej-afa-work'
